In [10]:
import os

RAW_DIR = "/mnt/d/matics/logs"
images_dirs = [
    os.path.join(RAW_DIR, "coco", "images"),
    os.path.join(RAW_DIR, "processed", "batch_01", "images"),
]
json_path = os.path.join(RAW_DIR, "processed", "batch_01", "annotations.json")
output_dir = os.path.join(RAW_DIR, "logs_600")

In [15]:
print(len(os.listdir(images_dirs[1])))

400


In [16]:
print(os.listdir(images_dirs[1]))

['0017.JPG', '0021.JPG', '0023.JPG', '0025.JPG', '0027.JPG', '0040.JPG', '0046.JPG', '0049.JPG', '0055.JPG', '0056.JPG', '0061.JPG', '0066.JPG', '0074.JPG', '0095.JPG', '0098.JPG', '0099.JPG', '0119.JPG', '0121.JPG', '0132.JPG', '0135.JPG', '0141.JPG', '0143.JPG', '0144.JPG', '0148.JPG', '0152.JPG', '0155.JPG', '0169.JPG', '0175.JPG', '0176.JPG', '0178.JPG', '0215.JPG', '0220.JPG', '0226.JPG', '0230.JPG', '0242.JPG', '0243.JPG', '0244.JPG', '0247.JPG', '0256.JPG', '0272.JPG', '0275.JPG', '0287.JPG', '0310.JPG', '0320.JPG', '0327.JPG', '0333.JPG', '0334.JPG', '0341.JPG', '0342.JPG', '0345.JPG', '0352.JPG', '0367.JPG', '0374.JPG', '0377.JPG', '0387.JPG', '0394.JPG', '0402.JPG', '0410.JPG', '0412.JPG', '0422.JPG', '0423.JPG', '0425.JPG', '0428.JPG', '0443.JPG', '0455.JPG', '0461.JPG', '0464.JPG', '0501.JPG', '0504.JPG', '0505.JPG', '0506.JPG', '0514.JPG', '0529.JPG', '0540.JPG', '0545.JPG', '0549.JPG', '0552.JPG', '0560.JPG', '0564.JPG', '0568.JPG', '0573.JPG', '0576.JPG', '0594.JPG', '05

In [ ]:
from pycocotools.coco import COCO
import json


def join_coco_batches(images_dirs: list[str], json_path: str, output_dir: str):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    coco = COCO(json_path)

    # Remove images that have no annotations from dataset, store new json
    coco_images = coco.loadImgs(coco.getImgIds())
    coco_images = [
        img for img in coco_images if len(coco.getAnnIds(imgIds=img["id"])) > 0
    ]
    # Print number of images before and after
    print(f"Number of images before: {len(coco.dataset['images'])}")
    print(f"Number of images after: {len(coco_images)}")
    coco.dataset["images"] = coco_images
    # check there are no annotations without images, if there are raise an error
    coco_anns = coco.loadAnns(coco.getAnnIds())
    coco_anns_checked = [ann for ann in coco_anns if coco.loadImgs(ann["image_id"])]
    if len(coco_anns) != len(coco_anns_checked):
        raise ValueError("Annotations without corresponding images found")
    # save new json
    with open(os.path.join(output_dir, "annotations.json"), "w") as f:
        json.dump(coco.dataset, f)

    images_filenames = [img["file_name"] for img in coco_images]
    for images_dir in images_dirs:
        for filename in os.listdir(images_dir):
            if filename in images_filenames:
                src = os.path.join(images_dir, filename)
                dst = os.path.join(output_dir, "images", filename)
                # copy
                os.system(f"cp {src} {dst}")

In [17]:
from concurrent.futures import ThreadPoolExecutor
import json
import os
import shutil
from pycocotools.coco import COCO
from tqdm import tqdm


def copy_image(src_dst_tuple):
    """Helper function to copy a single file."""
    src, dst = src_dst_tuple
    try:
        shutil.copy(src, dst)
    except Exception as e:
        print(f"Error copying {src} to {dst}: {e}")


def join_coco_batches(
    images_dirs: list[str], json_path: str, output_dir: str, max_workers: int = 8
):
    # Create output directories if they don't exist
    os.makedirs(output_dir, exist_ok=True)
    images_output_dir = os.path.join(output_dir, "images")
    os.makedirs(images_output_dir, exist_ok=True)

    coco = COCO(json_path)

    # Remove images that have no annotations from dataset, store new json
    coco_images = coco.loadImgs(coco.getImgIds())
    coco_images = [
        img for img in coco_images if len(coco.getAnnIds(imgIds=img["id"])) > 0
    ]

    # Print number of images before and after
    print(f"Number of images before: {len(coco.dataset['images'])}")
    print(f"Number of images after: {len(coco_images)}")
    coco.dataset["images"] = coco_images

    # Check there are no annotations without images
    coco_anns = coco.loadAnns(coco.getAnnIds())
    coco_anns_checked = [
        ann for ann in coco_anns if coco.loadImgs(ann["image_id"])
    ]
    if len(coco_anns) != len(coco_anns_checked):
        raise ValueError("Annotations without corresponding images found")

    # Save new json
    print("Saving filtered annotations JSON...")
    with open(os.path.join(output_dir, "annotations.json"), "w") as f:
        json.dump(coco.dataset, f)

    # Convert to a set for O(1) lookups instead of O(N) list searches
    images_filenames = {img["file_name"].lower() for img in coco_images}

    # Gather all copy tasks first
    print("Scanning directories for matching images...")
    copy_tasks = []
    for images_dir in images_dirs:
        if not os.path.exists(images_dir):
            continue
        for filename in os.listdir(images_dir):
            if filename.lower() in images_filenames:
                src = os.path.join(images_dir, filename)
                dst = os.path.join(images_output_dir, filename)
                copy_tasks.append((src, dst))

    # Multithreaded copying with a progress bar
    print(f"Copying {len(copy_tasks)} images using {max_workers} threads...")
    if copy_tasks:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # list() forces the generator to evaluate, ensuring tqdm updates
            list(
                tqdm(
                    executor.map(copy_image, copy_tasks),
                    total=len(copy_tasks),
                    desc="Copying images",
                )
            )
    else:
        print("No matching images found to copy.")


# Example usage:
# join_coco_batches(["/path/to/batch1", "/path/to/batch2"], "/path/to/instances.json", "/path/to/output")

In [18]:
join_coco_batches(images_dirs, json_path, output_dir)

loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
Number of images before: 800
Number of images after: 600
Saving filtered annotations JSON...
Scanning directories for matching images...
Copying 600 images using 8 threads...


Copying images: 100%|██████████| 600/600 [01:26<00:00,  6.92it/s]


In [19]:
len(os.listdir(os.path.join(output_dir, "images")))

600